# Meta-MedRAG — Gradio Interface

> **Prerequisite:** Run `Meta_MedRAG_A100_Final.ipynb`

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────
import sys, os, shutil
from pathlib import Path

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/meta_medrag_results')
REPO  = Path('/content/meta_medrag')

# Clone repo if needed
if not REPO.exists():
    os.system('git clone https://github.com/nourboudhina/meta-medrag.git /content/meta_medrag')
else:
    print('Repo already cloned')

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    'gradio==3.35.2', 'gradio-client==0.2.9',
    'loguru', 'pyyaml', 'Pillow',
    'open-clip-torch', 'faiss-cpu==1.8.0',
    'numpy==1.26.4',
], check=True)
print('Dependencies installed')

In [ ]:
# ── Cell 3: Sync checkpoints from Drive ──────────────────────
from pathlib import Path
import shutil

for src, dst in [
    (DRIVE/'checkpoints/meco_probe_v2.pkl',
     REPO/'checkpoints/meco_probe_v2.pkl'),
    (DRIVE/'vector_stores/radiology.index',
     REPO/'data/vector_stores/radiology.index'),
]:
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    if Path(src).exists():
        shutil.copy(src, dst)
        print(f'Synced: {Path(dst).name}')
    else:
        print(f'Missing in Drive: {Path(src).name}')

In [ ]:
# ── Cell 4: Load pipeline ────────────────────────────────────
import os
os.environ['MEDRAG_DATA_DIR'] = 'data/raw/iu_xray/images'

from src.pipeline import MetaMedRAGPipeline

pipeline = MetaMedRAGPipeline(
    config_path='configs/config.yaml',
    probe_path='checkpoints/meco_probe_v2.pkl',
)
print('Pipeline loaded')

In [ ]:
# ── Cell 5: Launch Gradio ─────────────────────────────────────
from src.interface.app import build_interface, load_pipeline

load_pipeline('configs/config.yaml')
demo = build_interface()

demo.launch(
    share=True,
    server_port=7860,
    server_name='0.0.0.0',
)